# Phase 6 — stub

Backbone: `Qwen/Qwen3-8B`, bf16, A100. Same weights as phases 3–5, so every layer index
from those phases carries over unchanged.

Nothing loaded but the model.

In [ ]:
# Setup: GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [ ]:
# Environment — must run BEFORE anything imports huggingface_hub.
#
# 1. HF_HUB_DISABLE_XET: the Xet backend hung this notebook dead — small JSON files
#    completed, then "Downloading bytes: 0.00B / Fetching 5 files: 0/5" sat at zero
#    for 5+ minutes on the safetensors shards. Same failure phase 3 hit; see
#    phase3/README.md's operational note. Falls back to plain HTTPS range requests.
# 2. HF_TOKEN: read from the Colab secrets vault. Needs this notebook's per-secret
#    "Notebook access" toggle ON (key icon, left sidebar), or the fetch blocks on a
#    grant prompt and times out with "Secrets can only be fetched when running from
#    the Colab UI". Unauthenticated works for public repos but is rate-limited.
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    print("HF_TOKEN present:", bool(tok))
except Exception as e:
    print(f"HF_TOKEN unavailable ({type(e).__name__}) — continuing unauthenticated")

print("HF_HUB_DISABLE_XET:", os.environ["HF_HUB_DISABLE_XET"])

HF_TOKEN present: True
HF_HUB_DISABLE_XET: 1


In [ ]:
# Load Qwen3-8B (bf16 where supported, else fp16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-8B"

# T4 is Turing (SM 7.5): no bf16. A100/L4 are Ampere+ and worth taking.
BF16  = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16, "| using", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map="cuda").eval()   # `torch_dtype` is deprecated in v5

print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)
print("layers:", model.config.num_hidden_layers, "| d_model:", model.config.hidden_size,
      "| vocab:", model.config.vocab_size)
print(f"weights: {sum(p.numel() for p in model.parameters())/1e9:.2f} B | "
      f"GPU total: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB | "
      f"allocated: {torch.cuda.memory_allocated()/2**30:.1f} GiB")

GPU: NVIDIA A100-SXM4-40GB | bf16: True | using torch.bfloat16


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded: Qwen/Qwen3-8B
device: cuda:0 | dtype: torch.bfloat16
layers: 36 | d_model: 4096 | vocab: 151936
weights: 8.19 B | GPU total: 39.5 GiB | allocated: 15.3 GiB


In [ ]:
# Sanity: one short greedy completion, thinking off.
# Note: in transformers 5.x apply_chat_template(return_tensors="pt") returns a
# BatchEncoding, not a tensor — so build the string, then encode it.
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "say hello in five words"}],
    add_generation_prompt=True, enable_thinking=False, tokenize=False)
enc = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**enc, max_new_tokens=32, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id)
print(repr(tokenizer.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True)))

'Hello, how can I help?'


In [ ]:
# === Expected cosine-to-'bridge' under the next-token distribution ===
#
#   score = SUM_v  p(v) * cos(e_v, e_bridge)
#
# p(v) is the full-vocab next-token distribution from ONE forward pass (the last
# prompt position, thinking off). Four spaces are scored because the choice is not
# obvious and it is cheap to do all of them:
#   in / out      input embeddings vs the untied lm_head rows
#   raw / cent    as-is vs mean-centred (phase 2's pool used ||E[i] - mean(E)||,
#                 so the embedding mean is known to carry weight on this backbone)
#
# Raw cosines are dominated by the shared mean direction, so the absolute number is
# meaningless on its own — the uniform-distribution baseline at the bottom is what
# each score has to be read against.
import torch, torch.nn.functional as F

QUERIES = [
    "what shall i do today",
    "recommend me a book",
    "how do I make friends in a new city?",
    "what should I get my brother for his birthday?",
    "how do I make friends in a new city?",
    # positive controls — these SHOULD score high if the measure means anything
    "tell me about bridges",
    "explain how suspension bridges work",
]
QUERIES = list(dict.fromkeys(QUERIES))   # dedupe, keep order

TARGET = " bridge"
tgt = tokenizer(TARGET, add_special_tokens=False).input_ids
assert len(tgt) == 1, f"{TARGET!r} is not a single token: {tgt}"
TGT_ID = tgt[0]
print(f"target {TARGET!r} -> id {TGT_ID}\n")

# --- cosine of every vocab item to the target, in each space ------------------
assert model.config.tie_word_embeddings is False, "embeddings are tied; in/out are identical"
SPACES = {"in": model.model.embed_tokens.weight, "out": model.lm_head.weight}

COS = {}
for name, W in SPACES.items():
    E = W.detach().float()                       # [V, d]
    for kind in ("raw", "cent"):
        X = E - E.mean(0, keepdim=True) if kind == "cent" else E
        Xn = F.normalize(X, dim=-1)
        COS[f"{name}.{kind}"] = (Xn @ Xn[TGT_ID]).contiguous()
        del Xn, X
    del E
    torch.cuda.empty_cache()

KEYS = list(COS)

# --- one forward pass per query, keep the full distribution -------------------
P = {}                                            # query -> [V] probabilities on CPU
rows = []
for q in QUERIES:
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        add_generation_prompt=True, enable_thinking=False, tokenize=False)
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1].float()
    p = logits.softmax(-1)
    P[q] = p.cpu()
    rows.append((q, [(p * COS[k]).sum().item() for k in KEYS], p))

# --- report -------------------------------------------------------------------
w = max(len(q) for q in QUERIES)
print(f"{'query':<{w}} " + " ".join(f"{k:>10}" for k in KEYS))
print("-" * (w + 11 * len(KEYS)))
for q, scores, _ in rows:
    print(f"{q:<{w}} " + " ".join(f"{s:>10.4f}" for s in scores))

print("-" * (w + 11 * len(KEYS)))
print(f"{'[uniform baseline]':<{w}} " + " ".join(f"{COS[k].mean().item():>10.4f}" for k in KEYS))
print(f"{'[self: p=1 on target]':<{w}} " + " ".join(f"{COS[k][TGT_ID].item():>10.4f}" for k in KEYS))

# --- what is actually driving the sum ----------------------------------------
print("\ntop-5 tokens by p * cos(in.cent), per query:")
for q, _, p in rows:
    contrib = p * COS["in.cent"]
    top = contrib.topk(5)
    print(f"\n  {q!r}")
    for c, i in zip(top.values.tolist(), top.indices.tolist()):
        print(f"    {tokenizer.decode([i])!r:<16} p={p[i].item():.4f} "
              f"cos={COS['in.cent'][i].item():+.4f} -> {c:+.5f}")

print(f"\nfull distributions kept in P (dict of {len(P)} tensors, {P[QUERIES[0]].shape[0]} entries each)")

target ' bridge' -> id 14164

query                                              in.raw    in.cent    out.raw   out.cent
------------------------------------------------------------------------------------------
what shall i do today                              0.0117    -0.0042    -0.0043     0.0318
recommend me a book                                0.0093    -0.0087     0.0081     0.0055
how do I make friends in a new city?               0.0259     0.0072     0.0209     0.0066
what should I get my brother for his birthday?     0.0229     0.0051     0.0160     0.0111
tell me about bridges                              0.0309     0.0124    -0.0245     0.0730
explain how suspension bridges work                0.0153    -0.0034     0.0643     0.0412
------------------------------------------------------------------------------------------
[uniform baseline]                                 0.0166    -0.0023     0.0184    -0.0002
[self: p=1 on target]                              1.0000   

In [ ]:
# === Same measure, over the whole answer ===
#
#   score_t = SUM_v  p_t(v) * cos(e_v, e_bridge)     for every answer position t
#
# Generate greedily, then one teacher-forced pass over prompt+completion to recover
# the full distribution at every position at once. Reported per query:
#   mean/max   the expected cosine, averaged over answer positions
#   realised   mean cos of the tokens actually emitted (what the argmax path scores)
#   H          mean entropy in bits — the first-pass measure failed because this was
#              under 1 bit, so the sum was just cos(argmax token, bridge)
import torch

MAX_NEW = 160
TRAJ = {}      # query -> [T] per-position scores in in.cent
ANS  = {}

summary = []
for q in QUERIES:
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        add_generation_prompt=True, enable_thinking=False, tokenize=False)
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    n_prompt = enc.input_ids.shape[1]

    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)[0]
    ans_ids = gen[n_prompt:]
    ANS[q] = tokenizer.decode(ans_ids, skip_special_tokens=True)

    # logits[i] predicts token i+1, so answer positions are n_prompt-1 .. len-2
    with torch.no_grad():
        logits = model(gen.unsqueeze(0)).logits[0, n_prompt - 1 : len(gen) - 1].float()
    Pa = logits.softmax(-1)                                   # [T, V]

    sc = {k: (Pa @ COS[k]) for k in KEYS}                     # [T] each
    TRAJ[q] = sc["in.cent"].cpu()
    H = -(Pa * Pa.clamp_min(1e-12).log2()).sum(-1)            # [T]
    realised = COS["in.cent"][ans_ids].mean().item()

    summary.append((q, {k: sc[k].mean().item() for k in KEYS},
                    sc["in.cent"].max().item(), realised, H.mean().item(), len(ans_ids)))
    del logits, Pa
    torch.cuda.empty_cache()

w = max(len(q) for q in QUERIES)
hdr = " ".join(f"{k:>9}" for k in KEYS)
print(f"{'query':<{w}} {hdr} {'max':>9} {'realised':>9} {'H bits':>7} {'T':>4}")
print("-" * (w + 10 * len(KEYS) + 33))
for q, means, mx, rl, h, T in summary:
    print(f"{q:<{w}} " + " ".join(f"{means[k]:>9.4f}" for k in KEYS)
          + f" {mx:>9.4f} {rl:>9.4f} {h:>7.2f} {T:>4}")
print("-" * (w + 10 * len(KEYS) + 33))
print(f"{'[uniform baseline]':<{w}} " + " ".join(f"{COS[k].mean().item():>9.4f}" for k in KEYS))

print("\nper-position trajectory (in.cent), every 8th token:")
for q in QUERIES:
    t = TRAJ[q]
    print(f"\n  {q!r}")
    print("   " + " ".join(f"{v:+.3f}" for v in t[::8].tolist()))

print("\nanswers (first 150 chars):")
for q in QUERIES:
    print(f"\n  {q!r}\n    {ANS[q][:150]!r}")

query                                             in.raw   in.cent   out.raw  out.cent       max  realised  H bits    T
-----------------------------------------------------------------------------------------------------------------------
what shall i do today                             0.0368    0.0198   -0.0216    0.0546    0.0712    0.0205    0.70  160
recommend me a book                               0.0359    0.0189   -0.0181    0.0516    0.0525    0.0186    0.53   94
how do I make friends in a new city?              0.0392    0.0222   -0.0158    0.0499    0.0753    0.0222    0.74  160
what should I get my brother for his birthday?    0.0405    0.0232   -0.0178    0.0534    0.0671    0.0234    0.73  160
tell me about bridges                             0.0549    0.0377    0.0137    0.0864    0.8962    0.0382    0.64  160
explain how suspension bridges work               0.0782    0.0615    0.0379    0.1025    1.0000    0.0564    0.65  160
----------------------------------------

In [ ]:
# === Steer the control queries in the bridge direction ===
#
# Phase 5's working setting, rebuilt: CAA mean over 8 `bridge - X` pairs, each arm
# carrying the shared context prefix so the differing token is NOT at position 0
# (bare single-token pairs land on Qwen3-8B's attention sink, where from L7 the two
# prompts are the same vector to 5 decimals — that artifact invalidated phase 4's
# layer curve). Inject at L16, strength 1.0 x the non-sink residual norm, every
# position except 0, held on during generation.
import torch, torch.nn.functional as F
from contextlib import contextmanager

LAYERS  = model.model.layers
L_STEER = 16
S       = 1.0
CTX     = "The word is"
POS     = " bridge"

def _ids(t): return tokenizer(t, add_special_tokens=False).input_ids

# arms must tokenize to equal length or their final tokens sit at different positions
_n = len(_ids(CTX + POS))
NEGS = [n for n in [" cat", " chair", " cloud", " music", " running",
                    " table", " coffee", " window", " paper", " orange"]
        if len(_ids(CTX + n)) == _n][:8]
print(f"pair: {POS!r} - {{{', '.join(repr(n) for n in NEGS)}}}  (ctx={CTX!r})")

@torch.no_grad()
def last_tok_vector(pos_txt, neg_txt, layer, ctx=CTX):
    a, b = _ids(ctx + pos_txt), _ids(ctx + neg_txt)
    ha = model(torch.tensor([a], device=model.device), output_hidden_states=True).hidden_states[layer][0]
    hb = model(torch.tensor([b], device=model.device), output_hidden_states=True).hidden_states[layer][0]
    return (ha[-1] - hb[-1]).float()

SINGLES = torch.stack([last_tok_vector(POS, n, L_STEER) for n in NEGS])
V_CAA   = SINGLES.mean(0)
_off    = F.normalize(SINGLES, dim=-1) @ F.normalize(SINGLES, dim=-1).T
_off    = _off[~torch.eye(len(NEGS), dtype=bool, device=_off.device)]
print(f"||v|| single mean {SINGLES.norm(dim=-1).mean():.1f} -> CAA {V_CAA.norm():.1f} | "
      f"pairwise cos {_off.mean():.3f}  (low = the negative arm is load-bearing)")

@torch.no_grad()
def nonsink_norm(text, layer):
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    h = model(**enc, output_hidden_states=True).hidden_states[layer][0].float()
    return h[1:].norm(dim=-1).mean().item()

@contextmanager
def steer_at(layer, v, alpha):
    """Add alpha*v at every position except 0, prompt and decode steps alike."""
    vv = (alpha * v).to(model.dtype)
    def hook(mod, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        h = h.clone()
        if h.shape[1] > 1: h[:, 1:] += vv        # prompt / teacher-forced pass
        else:              h += vv               # decode step, never position 0
        if args: return (h,) + tuple(args[1:]), kwargs
        kwargs["hidden_states"] = h
        return args, kwargs
    hd = LAYERS[layer].register_forward_pre_hook(hook, with_kwargs=True)
    try: yield
    finally: hd.remove()

def _chat(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)

@torch.no_grad()
def answer_and_score(q, steer=True, max_new=160):
    """Generate (steered or not), then score every answer position under the SAME
    condition, plus perplexity of the result under the UNSTEERED model."""
    enc = tokenizer(_chat(q), return_tensors="pt").to(model.device)
    n_p = enc.input_ids.shape[1]
    alpha = (S * nonsink_norm(_chat(q), L_STEER) / V_CAA.norm()).item() if steer else 0.0

    ctx = steer_at(L_STEER, V_CAA, alpha) if steer else torch.no_grad()
    with ctx:
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)[0]
        logits = model(gen.unsqueeze(0)).logits[0, n_p - 1 : len(gen) - 1].float()
    Pa = logits.softmax(-1)
    sc = {k: (Pa @ COS[k]).mean().item() for k in KEYS}

    ans = gen[n_p:]
    clean = model(gen.unsqueeze(0)).logits[0, n_p - 1 : len(gen) - 1].float()   # no steering
    ppl = torch.exp(F.cross_entropy(clean, ans)).item()
    return tokenizer.decode(ans, skip_special_tokens=True), sc, ppl, len(ans)

CONTROLS = [q for q in QUERIES if "bridge" not in q.lower()]
STEERED = {}
for q in CONTROLS:
    STEERED[q] = answer_and_score(q, steer=True)
    torch.cuda.empty_cache()

w = max(len(q) for q in CONTROLS)
print(f"\n{'query':<{w}} {'in.cent OFF':>12} {'in.cent ON':>11} {'ppl ON':>8}")
print("-" * (w + 34))
for q in CONTROLS:
    off = dict(zip(KEYS, [s for s in next(m for qq, m, *_ in summary if qq == q).values()]))
    print(f"{q:<{w}} {off['in.cent']:>12.4f} {STEERED[q][1]['in.cent']:>11.4f} {STEERED[q][2]:>8.1f}")

print("\n" + "=" * 100)
for q in CONTROLS:
    print(f"\n### {q!r}")
    print(f"  OFF: {ANS[q][:220]!r}")
    print(f"  ON : {STEERED[q][0][:220]!r}")

pair: ' bridge' - {' cat', ' chair', ' cloud', ' music', ' running', ' table', ' coffee', ' window'}  (ctx='The word is')
||v|| single mean 69.4 -> CAA 50.0 | pairwise cos 0.452  (low = the negative arm is load-bearing)

query                                           in.cent OFF  in.cent ON   ppl ON
--------------------------------------------------------------------------------
what shall i do today                                0.0198      0.3021      2.3
recommend me a book                                  0.0189      0.3108      2.9
how do I make friends in a new city?                 0.0222      0.5376      2.6
what should I get my brother for his birthday?       0.0232      0.3403      2.6


### 'what shall i do today'
  OFF: "That's a great question! What you do today depends on what you're interested in, what you need to accomplish, and how you want to feel. Here are a few ideas to help you decide:\n\n### 1. **Reflect and Plan**\n- **What do yo"
  ON : 'To the bridge, and the

In [ ]:
# === Strength sweep under phase 5's actual protocol ===
#
# The cell above used greedy / 160 tokens and looped. Phase 5 evaluated at T=0.8,
# 45 new tokens, and scored three numbers rather than one, because rate alone ranks
# 'bridge bridge bridge' at the top. Perplexity does NOT catch this failure — a
# repetition loop is highly predictable, so it scores LOW. Use a distinctness ratio.
import torch, torch.nn.functional as F

def distinct_ratio(ids):
    """unique tokens / total — 1.0 = no repeats, ~0.1 = locked in a loop."""
    return len(set(ids.tolist())) / max(1, len(ids))

@torch.no_grad()
def sample_steered(q, s, max_new=45, n=4, seed=0):
    enc   = tokenizer(_chat(q), return_tensors="pt").to(model.device)
    n_p   = enc.input_ids.shape[1]
    alpha = s * nonsink_norm(_chat(q), L_STEER) / V_CAA.norm().item() if s else 0.0
    outs, scores, dist = [], [], []
    for i in range(n):
        torch.manual_seed(seed + i)
        ctx = steer_at(L_STEER, V_CAA, alpha) if s else torch.no_grad()
        with ctx:
            gen = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                                 temperature=0.8, top_p=0.95,
                                 pad_token_id=tokenizer.eos_token_id)[0]
            lg = model(gen.unsqueeze(0)).logits[0, n_p - 1 : len(gen) - 1].float()
        ans = gen[n_p:]
        outs.append(tokenizer.decode(ans, skip_special_tokens=True))
        scores.append((lg.softmax(-1) @ COS["in.cent"]).mean().item())
        dist.append(distinct_ratio(ans))
    return outs, sum(scores) / n, sum(dist) / n

STRENGTHS = [0.0, 0.4, 0.6, 0.8, 1.0]
SWEEP = {}
print(f"{'query':<48} {'s':>4} {'in.cent':>8} {'distinct':>9}  sample")
print("-" * 130)
for q in CONTROLS:
    for s in STRENGTHS:
        outs, sc, dr = sample_steered(q, s)
        SWEEP[(q, s)] = (outs, sc, dr)
        flag = "  <-- looping" if dr < 0.45 else ""
        print(f"{q[:48]:<48} {s:>4.1f} {sc:>8.4f} {dr:>9.2f}  {outs[0][:52]!r}{flag}")
    print()
    torch.cuda.empty_cache()

query                                               s  in.cent  distinct  sample
----------------------------------------------------------------------------------------------------------------------------------
what shall i do today                             0.0   0.0177      0.84  "That's a great question! What you do today depends o"
what shall i do today                             0.4   0.0217      0.81  "That's a great question! It's always exciting to thi"
what shall i do today                             0.6   0.0302      0.80  "Ah, the classic question! It's a gateway to many pos"
what shall i do today                             0.8   0.0972      0.54  'To bridge the gap between what is and what is not. Y'
what shall i do today                             1.0   0.1300      0.40  'To the unknown, I stand. You do not know the bridge,'  <-- looping

recommend me a book                               0.0   0.0194      0.88  "Sure! Here's a great book recommendation based on yo"


In [ ]:
# === Phase 6 §1 — GCG machinery aimed at the bridgeness metric ===
#
# Objective IS the metric: mean over answer positions of SUM_v p_t(v)*cos(e_v, e_bridge),
# in the in.cent space. Not a proxy for it — the thing itself. The answer is a greedy
# rollout from the current trigger, refreshed every `refresh_every` steps; the gradient
# and the accept test both score against that rollout, and a separate `true_metric`
# regenerates from scratch so progress is never read off the teacher-forced number alone.
#
# Blocklist ON (bridge in ~40 languages + top-300 embedding neighbours). Without it a
# 254-slot search just writes an English prompt injection, which measures nothing.
# Pictographs ALLOWED.
import torch, torch.nn.functional as F, random, time, inspect, unicodedata

dev = model.device
V   = model.config.vocab_size
TARGET_COS = COS["in.cent"]                     # the metric's cosine vector
_LTK = ("logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters
        else "num_logits_to_keep")

# ---------- vocab strings, decoded once ----------------------------------------
TOKSTR = tokenizer.batch_decode([[i] for i in range(V)])
NFKD   = [unicodedata.normalize("NFKD", s).casefold() for s in TOKSTR]

# ---------- structural guard ---------------------------------------------------
usable = torch.ones(V, dtype=torch.bool)
for i in set(tokenizer.all_special_ids) | set(tokenizer.get_added_vocab().values()):
    if i < V: usable[i] = False
for i, s in enumerate(TOKSTR):
    if not s.strip() or any(unicodedata.category(c) in ("Cc", "Cs", "Co") for c in s):
        usable[i] = False
print(f"vocab {V} -> usable {int(usable.sum())}")

# ---------- target blocklist ---------------------------------------------------
TRANSLATIONS = [
    "bridge", "bridges", "puente", "ponte", "pont", "brucke", "brücke", "brug", "bro",
    "brú", "мост", "міст", "most", "γέφυρα", "gefyra", "köprü", "kopru", "جسر", "גשר",
    "پل", "पुल", "সেতু", "桥", "橋", "大桥", "ブリッジ", "はし", "다리", "브리지",
    "cầu", "cau", "สะพาน", "jembatan", "jambatan", "silta", "sild", "híd", "hid",
    "pod", "tilts", "tiltas", "droichead", "pons", "ponto", "daraja", "tulay",
    "ხიდი", "կամուրջ", "viaduct", "viaduc", "aqueduct", "overpass", "causeway",
    "trestle", "footbridge",
]
blocked = torch.zeros(V, dtype=torch.bool)
for i, s in enumerate(NFKD):
    if s and any(t in s for t in TRANSLATIONS):
        blocked[i] = True
n_sub = int(blocked.sum())

E_c = (model.model.embed_tokens.weight.float()
       - model.model.embed_tokens.weight.float().mean(0, keepdim=True))
E_cn = F.normalize(E_c, dim=-1)
nbrs = (E_cn @ E_cn[TGT_ID]).topk(300).indices
blocked[nbrs.cpu()] = True
del E_cn
torch.cuda.empty_cache()
print(f"blocked: {n_sub} by substring (~{len(TRANSLATIONS)} forms) + neighbours "
      f"-> {int(blocked.sum())} total ({100*int(blocked.sum())/V:.2f}% of vocab)")

WEAKNESS = E_c.norm(dim=-1).cpu()               # small norm = undertrained
del E_c
torch.cuda.empty_cache()

def build_pool(kind):
    ok = usable & ~blocked
    if kind == "full":
        return ok.clone()
    n = int(kind.replace("weak", ""))
    idx = torch.nonzero(ok).squeeze(-1)
    keep = idx[WEAKNESS[idx].argsort()[:n]]
    m = torch.zeros(V, dtype=torch.bool); m[keep] = True
    return m

# ---------- scaffold -----------------------------------------------------------
SENT = "␞"
def make_scaffold(q, position):
    content = f"{SENT} {q}" if position == "prefix" else f"{q} {SENT}"
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": content}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)
    pre, suf = text.split(SENT)
    return (tokenizer(pre, add_special_tokens=False).input_ids,
            tokenizer(suf, add_special_tokens=False).input_ids)

# ---------- objective ----------------------------------------------------------
@torch.no_grad()
def rollout(trig, PRE, SUF, n_new):
    ids = torch.tensor([PRE + trig.tolist() + SUF], device=dev)
    out = model.generate(ids, max_new_tokens=n_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)[0]
    return out[ids.shape[1]:]

@torch.no_grad()
def score_batch(trigs, PRE, SUF, ans, chunk=8):
    """mean_t SUM_v p_t(v) cos_v, teacher-forced on `ans`. trigs: [B, k] -> [B]"""
    B  = trigs.shape[0]
    na = len(ans)
    pre = torch.tensor(PRE, device=dev); suf = torch.tensor(SUF, device=dev)
    out = []
    for i in range(0, B, chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b, -1), tb, suf.expand(b, -1),
                         ans.expand(b, -1)], dim=1)
        lg = model(seq, **{_LTK: na + 1}).logits[:, :-1].float()
        out.append((lg.softmax(-1) @ TARGET_COS).mean(-1))
        del lg
    return torch.cat(out)

def grad_onehot(trig, PRE, SUF, ans):
    E  = model.model.embed_tokens.weight
    oh = F.one_hot(trig.to(dev), num_classes=V).to(E.dtype).requires_grad_(True)
    inp = torch.cat([E[torch.tensor(PRE, device=dev)], oh @ E,
                     E[torch.tensor(SUF, device=dev)], E[ans]]).unsqueeze(0)
    lg = model(inputs_embeds=inp, **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    (lg.softmax(-1) @ TARGET_COS).mean().backward()
    g = oh.grad.detach().clone()
    del oh, inp, lg
    torch.cuda.empty_cache()
    return g

@torch.no_grad()
def true_metric(trig, PRE, SUF, n_new=45):
    ans = rollout(trig, PRE, SUF, n_new)
    return score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item(), ans

# ---------- the search ---------------------------------------------------------
def gcg(q="what shall i do today", k=32, position="suffix", pool_kind="weak4096",
        n_top=256, n_cand=128, n_new=45, refresh_every=4, chunk=8,
        steps=10**9, budget_s=90, seed=1, log=None):
    rng = random.Random(seed); torch.manual_seed(seed)
    PRE, SUF = make_scaffold(q, position)
    pool = build_pool(pool_kind)
    pidx = torch.nonzero(pool).squeeze(-1)
    trig = pidx[torch.randint(len(pidx), (k,), generator=torch.Generator().manual_seed(seed))]

    ans = rollout(trig, PRE, SUF, n_new)
    best_t, best_true = trig.clone(), score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item()
    t0, s = time.time(), 0
    while s < steps and time.time() - t0 < budget_s:
        g = grad_onehot(trig, PRE, SUF, ans)
        g[:, ~pool.to(dev)] = -float("inf")
        top = g.topk(min(n_top, int(pool.sum())), dim=-1).indices
        del g
        slots = torch.randint(k, (n_cand,))
        picks = top[slots, torch.randint(top.shape[1], (n_cand,))]
        cands = trig.unsqueeze(0).repeat(n_cand, 1).to(dev)
        cands[torch.arange(n_cand), slots] = picks
        sc = score_batch(cands, PRE, SUF, ans, chunk=chunk)
        j = int(sc.argmax())
        trig = cands[j].cpu()
        s += 1
        if s % refresh_every == 0:
            ans = rollout(trig, PRE, SUF, n_new)
            tm, _ = score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item(), None
            if tm > best_true: best_true, best_t = tm, trig.clone()
            if log: log(s, tm, time.time() - t0)
        del cands, sc, top
        torch.cuda.empty_cache()
    tm, a = true_metric(best_t, PRE, SUF, n_new)
    return dict(trigger=best_t, true=tm, answer=tokenizer.decode(a, skip_special_tokens=True),
                steps=s, secs=time.time() - t0)

print("machinery ready")

vocab 151936 -> usable 148023
blocked: 375 by substring (~55 forms) + neighbours -> 659 total (0.43% of vocab)
machinery ready


In [20]:
# === Phase 6 §2 — multi-slot GCG + Optuna hyperparameter search ===
#
# The single-substitution variant moved the metric from 0.0177 to ~0.019 in 20 steps at
# k=128 — one token per step cannot fill a long trigger. n_mut flips several slots per
# candidate. Everything else is unchanged: objective IS the metric, rollout refreshed
# every `refresh_every` steps, every candidate verified by a real forward pass.
!pip install -q optuna
import optuna, torch, time, random
optuna.logging.set_verbosity(optuna.logging.WARNING)

def gcg2(q="what shall i do today", k=64, position="suffix", pool_kind="weak4096",
         n_top=256, n_cand=128, n_mut=1, n_new=32, refresh_every=2, chunk=16,
         budget_s=150, seed=1, init="random", log=None):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    PRE, SUF = make_scaffold(q, position)
    pool = build_pool(pool_kind); pool_d = pool.to(dev)
    pidx = torch.nonzero(pool).squeeze(-1)
    if init == "repeat":
        trig = pidx[torch.randint(len(pidx), (1,), generator=gen)].repeat(k)
    else:
        trig = pidx[torch.randint(len(pidx), (k,), generator=gen)]

    ans = rollout(trig, PRE, SUF, n_new)
    best_t = trig.clone()
    best   = score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item()
    t0, s = time.time(), 0
    while time.time() - t0 < budget_s:
        g = grad_onehot(trig, PRE, SUF, ans)
        g[:, ~pool_d] = -float("inf")
        top = g.topk(min(n_top, int(pool.sum())), dim=-1).indices
        del g
        cands = trig.unsqueeze(0).repeat(n_cand, 1).to(dev)
        for _ in range(n_mut):
            slots = torch.randint(k, (n_cand,))
            picks = top[slots, torch.randint(top.shape[1], (n_cand,))]
            cands[torch.arange(n_cand), slots] = picks
        sc = score_batch(cands, PRE, SUF, ans, chunk=chunk)
        trig = cands[int(sc.argmax())].cpu()
        s += 1
        if s % refresh_every == 0:
            ans = rollout(trig, PRE, SUF, n_new)
            tm = score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item()
            if tm > best: best, best_t = tm, trig.clone()
            if log: log(s, tm, best, time.time() - t0)
        del cands, sc, top
        torch.cuda.empty_cache()
    tm, a = true_metric(best_t, PRE, SUF, 45)
    return dict(trigger=best_t, best_tf=best, true=tm, steps=s,
                answer=tokenizer.decode(a, skip_special_tokens=True))

TRIAL_S = 150
def objective(t):
    p = dict(
        k             = t.suggest_int("k", 16, 254, log=True),
        n_mut         = t.suggest_int("n_mut", 1, 8),
        n_top         = t.suggest_categorical("n_top", [64, 128, 256, 512, 1024]),
        n_cand        = t.suggest_categorical("n_cand", [64, 128, 256]),
        pool_kind     = t.suggest_categorical("pool_kind", ["weak4096", "weak16384", "full"]),
        position      = t.suggest_categorical("position", ["prefix", "suffix"]),
        n_new         = t.suggest_categorical("n_new", [24, 32, 48]),
        refresh_every = t.suggest_int("refresh_every", 1, 6),
        init          = t.suggest_categorical("init", ["random", "repeat"]),
    )
    try:
        r = gcg2(budget_s=TRIAL_S, seed=1, **p)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); raise optuna.TrialPruned()
    t.set_user_attr("trigger", tokenizer.decode(r["trigger"]))
    t.set_user_attr("answer", r["answer"][:300])
    t.set_user_attr("steps", r["steps"])
    print(f"  trial {t.number:>2}  true={r['true']:.4f}  steps={r['steps']:>3}  "
          f"k={p['k']:>3} mut={p['n_mut']} cand={p['n_cand']} pool={p['pool_kind']:<10} "
          f"{p['position']}")
    return r["true"]

N_TRIALS = 14
print(f"Optuna: {N_TRIALS} trials x {TRIAL_S}s ~= {N_TRIALS*TRIAL_S/60:.0f} min\n"
      f"reference points: unsteered 0.0177 | real bridge query 0.038-0.062 | "
      f"steering vector s=1.0 0.11-0.17\n")
study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=1))
study.optimize(objective, n_trials=N_TRIALS)

print(f"\nbest true metric: {study.best_value:.4f}")
print("best params:", study.best_params)
print("trigger:", repr(study.best_trial.user_attrs["trigger"][:200]))
print("answer :", repr(study.best_trial.user_attrs["answer"][:200]))

reference points, out.cent:
  uniform baseline                 -0.0002
  unsteered controls               +0.0499 .. +0.0546
  real bridge queries              +0.0864 .. +0.1025
  steering vector s=0.8            +0.0963 .. +0.1285
  steering vector s=1.0            +0.2066 .. +0.2277

Optuna: 14 trials x 150s ~= 35 min

  trial  0  true=0.0572  steps= 42  k= 50 mut=6 cand=256 top= 128 pool=weak16384  prefix init=repeat
  trial  1  true=0.0593  steps= 43  k=233 mut=3 cand=128 top= 256 pool=weak16384  prefix init=random
  trial  2  true=0.0615  steps= 30  k= 34 mut=7 cand=256 top= 256 pool=full       suffix init=repeat
  trial  3  true=0.0569  steps= 96  k= 49 mut=1 cand=64 top= 512 pool=weak4096   prefix init=random
  trial  4  true=0.0580  steps=172  k= 33 mut=8 cand=64 top= 128 pool=full       prefix init=repeat
  trial  5  true=0.0600  steps= 64  k= 89 mut=1 cand=64 top= 256 pool=full       prefix init=random
  trial  6  true=0.0616  steps= 52  k= 73 mut=7 cand=128 top= 512 pool=fu

In [24]:
# === Phase 6 §6 — post-study analysis, four-space control, and save ===
import torch, torch.nn.functional as F, json, time

# ---- 1. raw vs cent: equivalent for measurement, divergent in the tail? --------
# On natural text the two differ by a near-constant (~0.017 for `in`). The claim is
# that an optimiser destroys that equivalence, because the tail of cos_raw is tokens
# aligned with the vocabulary mean — generic rather than bridge-like. If the top-500
# sets barely overlap, the two objectives genuinely diverge where a search lives.
print("raw vs cent, across all 151,936 tokens:")
print(f"{'family':>6} {'pearson':>9} {'spearman':>9} {'top-500 overlap':>16} {'top-50':>8}")
for fam in ("in", "out"):
    r, c = COS[f"{fam}.raw"].float(), COS[f"{fam}.cent"].float()
    pear = torch.corrcoef(torch.stack([r, c]))[0, 1].item()
    rr = r.argsort().argsort().float(); cc = c.argsort().argsort().float()
    spear = torch.corrcoef(torch.stack([rr, cc]))[0, 1].item()
    t5r, t5c = set(r.topk(500).indices.tolist()), set(c.topk(500).indices.tolist())
    t50r, t50c = set(r.topk(50).indices.tolist()), set(c.topk(50).indices.tolist())
    print(f"{fam:>6} {pear:>9.4f} {spear:>9.4f} {len(t5r & t5c)/500:>15.1%} "
          f"{len(t50r & t50c)/50:>7.0%}")

print("\ntop-10 tokens by cos to ' bridge', each space:")
for k in KEYS:
    toks = [repr(tokenizer.decode([i])) for i in COS[k].topk(10).indices.tolist()]
    print(f"  {k:<9} {' '.join(toks)}")

# ---- 2. final run at the best params, keeping the trigger ids -----------------
bp = dict(study.best_params)
print(f"\nfinal run at best params: {bp}")
final = gcg2(budget_s=180, seed=1, **bp)
print(f"  true ({METRIC_SPACE}) = {final['true']:.4f}   steps={final['steps']}")

# ---- 3. the four-space control (RECIPE stage 6) -------------------------------
PRE, SUF = make_scaffold("what shall i do today", bp["position"])
ansf = rollout(final["trigger"], PRE, SUF, 45)
seq  = torch.cat([torch.tensor(PRE, device=dev), final["trigger"].to(dev),
                  torch.tensor(SUF, device=dev), ansf])
with torch.no_grad():
    lg = model(seq.unsqueeze(0), **{_LTK: len(ansf) + 1}).logits[0, :-1].float()
Pa = lg.softmax(-1)
four = {k: (Pa @ COS[k]).mean().item() for k in KEYS}
dist = len(set(ansf.tolist())) / len(ansf)

print(f"\nGCG winner scored in every space   (distinct={dist:.2f})")
print(f"{'space':>9} {'GCG':>9} {'unsteered ctrl':>15} {'real bridge q':>14}")
for k in KEYS:
    c = [m[k] for q, m, *_ in summary if "bridge" not in q.lower()]
    b = [m[k] for q, m, *_ in summary if "bridge" in q.lower()]
    print(f"{k:>9} {four[k]:>9.4f} {min(c):>7.4f}..{max(c):<6.4f} {min(b):>6.4f}..{max(b):<6.4f}")
print(f"\nanswer: {final['answer'][:300]!r}")

# ---- 4. save everything -------------------------------------------------------
out = dict(
    meta=dict(model=MODEL_ID, metric_space=METRIC_SPACE, thinking=False,
              layers=model.config.num_hidden_layers, vocab=V,
              blocked_tokens=int(blocked.sum()), usable=int(usable.sum())),
    first_pass={q: {k: s for k, s in zip(KEYS, sc)} for q, sc, _ in rows},
    whole_answer={q: dict(means=m, max=mx, realised=rl, entropy=h, T=T)
                  for q, m, mx, rl, h, T in summary},
    answers_unsteered={q: ANS[q] for q in ANS},
    steering=dict(
        layer=L_STEER, negatives=NEGS, ctx=CTX,
        caa_norm=float(V_CAA.norm()), pairwise_cos=float(_off.mean()),
        sweep={f"{q}|s={s}": dict(score=sc, distinct=dr, samples=o)
               for (q, s), (o, sc, dr) in SWEEP.items()}),
    gcg=dict(
        best_value=study.best_value, best_params=bp,
        trials=[dict(number=t.number, value=t.value, params=t.params,
                     trigger=t.user_attrs.get("trigger"),
                     answer=t.user_attrs.get("answer")) for t in study.trials],
        final=dict(true=final["true"], steps=final["steps"],
                   trigger_ids=final["trigger"].tolist(),
                   trigger=tokenizer.decode(final["trigger"]),
                   answer=final["answer"], distinct=dist, all_spaces=four)),
)
with open("/content/phase6_results.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
import os
print(f"\nsaved /content/phase6_results.json "
      f"({os.path.getsize('/content/phase6_results.json')/1024:.0f} kB)")

raw vs cent, across all 151,936 tokens:
family   pearson  spearman  top-500 overlap   top-50
    in    0.9851    0.9844           91.4%     96%
   out    0.5206    0.4729           49.0%     88%

top-10 tokens by cos to ' bridge', each space:
  in.raw    ' bridge' ' Bridge' ' bridges' 'bridge' 'Bridge' ' brid' ' Bridges' 'IDGE' '橋' ' projects'
  in.cent   ' bridge' ' Bridge' ' bridges' 'bridge' 'Bridge' ' brid' ' Bridges' 'IDGE' '橋' ' projects'
  out.raw   ' bridge' ' Bridge' ' bridges' '桥' 'bridge' 'Bridge' '橋' '桥梁' ' brid' '_bridge'
  out.cent  ' bridge' ' Bridge' ' bridges' '桥' 'bridge' 'Bridge' '橋' ' brid' '桥梁' '_bridge'

final run at best params: {'k': 53, 'n_mut': 7, 'n_top': 512, 'n_cand': 256, 'pool_kind': 'full', 'position': 'suffix', 'n_new': 48, 'refresh_every': 2, 'init': 'repeat'}
  true (out.cent) = 0.0620   steps=43

GCG winner scored in every space   (distinct=0.78)
    space       GCG  unsteered ctrl  real bridge q
   in.raw    0.0386  0.0359..0.0405 0.0549..0.0782
  i

In [22]:
import sys
for name in ["model", "COS", "study", "SWEEP", "summary", "V_CAA", "TARGET_COS"]:
    print(f"{name:<12}", "alive" if name in globals() else "GONE")
try:
    import torch
    print("cuda:", torch.cuda.get_device_name(0),
          f"| allocated {torch.cuda.memory_allocated()/2**30:.1f} GiB")
    print("trials completed:", len(study.trials))
    print("best so far:", round(study.best_value, 4), study.best_params)
except Exception as e:
    print("ERR", type(e).__name__, e)

model        alive
COS          alive
study        alive
SWEEP        alive
summary      alive
V_CAA        alive
TARGET_COS   alive
cuda: NVIDIA A100-SXM4-40GB | allocated 32.9 GiB
trials completed: 14
best so far: 0.062 {'k': 53, 'n_mut': 7, 'n_top': 512, 'n_cand': 256, 'pool_kind': 'full', 'position': 'suffix', 'n_new': 48, 'refresh_every': 2, 'init': 'repeat'}
